In [1]:
# filter_and_rebuild_cv_folds_2cls.py
#
# Purpose: Rebuild CV fold structure for 2-class training (deciduous_tree, shrub).
# Input:   labels_filtered/ — coniferous removed, classes remapped (1->0, 2->1)
# Input:   merged/images/   — unchanged tile images
# Input:   label_centroids_folds.csv, test_split.csv — spatial CV assignments
# Output:  cv_folds_2cls/   — same fold structure as cv_folds but with 2-class labels
# Note:    dataset.yaml files use DSS paths for cluster training
# Note:    Original labels/ folder is never modified

import os
import shutil
import pandas as pd

FOLDS_CSV  = r"F:\Masterthesis\dataset\merged\label_centroids_folds.csv"
TEST_CSV   = r"F:\Masterthesis\dataset\merged\test_split.csv"
IMAGES_DIR = r"F:\Masterthesis\dataset\merged\images"
LABELS_DIR = r"F:\Masterthesis\dataset\merged\labels_filtered"
OUTPUT_DIR = r"F:\Masterthesis\dataset\cv_folds_2cls"
CLASS_NAMES = ["deciduous_tree", "shrub"]

folds_df = pd.read_csv(FOLDS_CSV)
test_df  = pd.read_csv(TEST_CSV)

for fold_num in range(1, 6):
    for split in ["train", "val"]:
        os.makedirs(os.path.join(OUTPUT_DIR, f"fold{fold_num}", split, "images"), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, f"fold{fold_num}", split, "labels"), exist_ok=True)

os.makedirs(os.path.join(OUTPUT_DIR, "test", "images"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "test", "labels"), exist_ok=True)

def copy_tile(stem, dest_images, dest_labels):
    img_src = os.path.join(IMAGES_DIR, f"{stem}.png")
    lbl_src = os.path.join(LABELS_DIR, f"{stem}.txt")
    if os.path.exists(img_src):
        shutil.copy(img_src, os.path.join(dest_images, f"{stem}.png"))
    else:
        print(f"WARNING image not found: {stem}")
    if os.path.exists(lbl_src):
        shutil.copy(lbl_src, os.path.join(dest_labels, f"{stem}.txt"))
    else:
        print(f"WARNING label not found: {stem}")

for fold_num in range(1, 6):
    val_tiles   = folds_df[folds_df["fold"] == fold_num]
    train_tiles = folds_df[folds_df["fold"] != fold_num]
    for _, row in train_tiles.iterrows():
        copy_tile(row["filename"],
                  os.path.join(OUTPUT_DIR, f"fold{fold_num}", "train", "images"),
                  os.path.join(OUTPUT_DIR, f"fold{fold_num}", "train", "labels"))
    for _, row in val_tiles.iterrows():
        copy_tile(row["filename"],
                  os.path.join(OUTPUT_DIR, f"fold{fold_num}", "val", "images"),
                  os.path.join(OUTPUT_DIR, f"fold{fold_num}", "val", "labels"))

for _, row in test_df.iterrows():
    copy_tile(row["filename"],
              os.path.join(OUTPUT_DIR, "test", "images"),
              os.path.join(OUTPUT_DIR, "test", "labels"))

DSS_BASE = "/dss/dssfs04/lwp-dss-0002/pn67pu/pn67pu-dss-0000/ra26fuz/data/cv_folds_2cls"

for fold_num in range(1, 6):
    fold_dir_dss = f"{DSS_BASE}/fold{fold_num}"
    yaml_content = f"""path: {fold_dir_dss}
train: {fold_dir_dss}/train/images
val: {fold_dir_dss}/val/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
    with open(os.path.join(OUTPUT_DIR, f"fold{fold_num}", "dataset.yaml"), "w") as f:
        f.write(yaml_content)

test_yaml = f"""path: {DSS_BASE}/test
train: null
val: {DSS_BASE}/test/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
with open(os.path.join(OUTPUT_DIR, "test", "dataset.yaml"), "w") as f:
    f.write(test_yaml)

print("\nDone.")
for fold_num in range(1, 6):
    n_img = len(os.listdir(os.path.join(OUTPUT_DIR, f"fold{fold_num}", "train", "images")))
    n_lbl = len(os.listdir(os.path.join(OUTPUT_DIR, f"fold{fold_num}", "train", "labels")))
    print(f"fold{fold_num} train: {n_img} images, {n_lbl} labels")
test_n = len(os.listdir(os.path.join(OUTPUT_DIR, "test", "images")))
print(f"test: {test_n} images")


Done.
fold1 train: 264 images, 264 labels
fold2 train: 264 images, 264 labels
fold3 train: 266 images, 266 labels
fold4 train: 268 images, 268 labels
fold5 train: 262 images, 262 labels
test: 59 images
